# Week 13 — Google Earth Engine & Cloud-Scale Time Series
## ARIA v9.0 — Student Exercise Notebook (學生練習版)

**Course (課程):** Remote Sensing and Spatial Information Analysis and Applications
**Theme (主題):** Cloud-based satellite time-series analysis with Google Earth Engine
**Study Area (研究區域):** Hualien, Taiwan (花蓮) — Post-earthquake (2024/04/03) & landslide dam (堰塞湖) analysis
**Bounding Box:** `[121.2574, 23.6546, 121.4984, 23.7447]`

---

### Learning Objectives (學習目標)
1. Access and filter large satellite archives via GEE Python API
2. Compute NDVI time series and detect vegetation change after an earthquake
3. Integrate optical (Sentinel-2) and SAR (Sentinel-1) observations
4. Create pre/post composites and compute change maps
5. Export cloud-computed products for local analysis

### How to Use This Notebook (使用說明)
- Cells marked **COMPLETE** are ready to run — do not modify them.
- Cells with `# TODO:` require you to **fill in the blanks** before running.
- `# HINT:` comments provide guidance for each exercise.
- Run cells **in order** from top to bottom.

### Prerequisites (先備知識)
- W8: NDVI fundamentals — W9: Change detection — W10: SAR basics — W12: Classification

> **Student Exercise Notebook (學生練習筆記本)** — fill in the blanks and run.

---
## S1 — Environment Setup (環境設定) ✅ COMPLETE

Install and import the required packages. We authenticate with Google Earth Engine,
set up Chinese font rendering for matplotlib, define our Area of Interest (AOI),
and run a quick connectivity test.

> **Note:** You need a GEE-enabled Google Cloud project. Replace `'your-project-id'`
> with your actual project ID.

This cell is **complete** — just run it.

In [ ]:
# ============================================================
# S1 — Environment Setup (環境設定) — COMPLETE
# ============================================================
import warnings
warnings.filterwarnings('ignore')

# --- Core imports ---
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from datetime import datetime, timedelta
import os, platform

# --- GEE Authentication & Initialization ---
# ee.Authenticate()  # Uncomment on first run — opens a browser auth flow
ee.Initialize(project='your-project-id')

# --- Chinese font setup (跨平台中文字型) ---
from matplotlib import font_manager as fm

def setup_chinese_font():
    """Cross-platform Chinese font configuration."""
    system = platform.system()
    candidates = {
        'Windows': ['Microsoft JhengHei', 'Microsoft YaHei', 'SimHei'],
        'Darwin':  ['PingFang TC', 'Heiti TC', 'STHeiti'],
        'Linux':   ['Noto Sans CJK TC', 'WenQuanYi Micro Hei',
                    'AR PL UMing TW', 'Noto Sans TC']
    }
    available = {f.name for f in fm.fontManager.ttflist}
    for name in candidates.get(system, candidates['Linux']):
        if name in available:
            plt.rcParams['font.sans-serif'] = [name] + plt.rcParams['font.sans-serif']
            plt.rcParams['axes.unicode_minus'] = False
            print(f"  Chinese font set to: {name}")
            return name
    # Fallback: try all candidates
    for flist in candidates.values():
        for name in flist:
            if name in available:
                plt.rcParams['font.sans-serif'] = [name] + plt.rcParams['font.sans-serif']
                plt.rcParams['axes.unicode_minus'] = False
                print(f"  Chinese font (fallback) set to: {name}")
                return name
    print("  WARNING: No CJK font found — Chinese text may not render correctly.")
    return None

font_name = setup_chinese_font()

# --- Define AOI (定義研究區域) ---
HUALIEN_BBOX = [121.2574, 23.6546, 121.4984, 23.7447]
aoi = ee.Geometry.Rectangle(HUALIEN_BBOX)

# --- Quick connectivity test (連線測試) ---
point = ee.Geometry.Point([121.37, 23.70])
elev = ee.Image('USGS/SRTMGL1_003').sample(point, 30).first().get('elevation').getInfo()
print(f"  Connectivity OK — Elevation at test point: {elev} m")

print()
print("=" * 60)
print("  S1 COMPLETE — Environment ready")
print(f"  AOI: {HUALIEN_BBOX}")
print(f"  Platform: {platform.system()} / Python {platform.python_version()}")
print("=" * 60)

---
## S2 — Filter Sentinel-2 ImageCollection (Sentinel-2 影像集篩選) ✏️ EXERCISE

We access the **Sentinel-2 Level-2A Surface Reflectance (Harmonized)** collection
and filter it by:
- **Date range (日期範圍):** 2020-01-01 to 2026-03-31
- **Cloud cover (雲量):** less than 40%
- **Spatial bounds (空間範圍):** our Hualien AOI

### Key GEE Concepts (關鍵概念)
- `ee.ImageCollection` — a stack of images managed on the server
- `filterDate(start, end)` — keep only images within a date range
- `filterBounds(geometry)` — keep only images that overlap our AOI
- `filter(ee.Filter.lt(property, value))` — filter by metadata property

> **HINT:** See Pre-lab Step 4 for ImageCollection filtering concepts.

In [ ]:
# ============================================================
# S2 — Filter Sentinel-2 ImageCollection — EXERCISE
# ============================================================

s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          # TODO: Filter by date range — 2020-01-01 to 2026-03-31
          # HINT: .filterDate('start_date', 'end_date')
          .filterDate('____', '____')
          # TODO: Filter by cloud cover — keep images with CLOUDY_PIXEL_PERCENTAGE < 40
          # HINT: ee.Filter.lt('property_name', threshold_value)
          .filter(ee.Filter.lt('____', ____))
          .filterBounds(aoi))

# --- Collection statistics (do not modify below) ---
count = s2_col.size().getInfo()
print(f"  Total Sentinel-2 images (2020-2026, cloud < 40%): {count}")

# Date range
dates = s2_col.aggregate_array('system:time_start').getInfo()
dates_dt = [datetime.utcfromtimestamp(d / 1000) for d in dates]
print(f"  Date range: {min(dates_dt).strftime('%Y-%m-%d')} -> {max(dates_dt).strftime('%Y-%m-%d')}")

# Cloud cover stats
cloud_stats = s2_col.aggregate_stats('CLOUDY_PIXEL_PERCENTAGE').getInfo()
print(f"  Cloud cover — mean: {cloud_stats['mean']:.1f}%, "
      f"min: {cloud_stats['min']:.1f}%, max: {cloud_stats['max']:.1f}%")

# --- Display first image as RGB ---
first_img = s2_col.sort('system:time_start').first()
vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}

Map2 = geemap.Map(center=[23.70, 121.40], zoom=11)
Map2.addLayer(first_img.clip(aoi), vis_rgb, 'First S2 Image (RGB)')
Map2.addLayer(aoi, {'color': 'red'}, 'AOI — Hualien')
Map2

---
## S3 — NDVI Calculation + Cloud Masking (NDVI 計算與雲遮罩) ✏️ EXERCISE

Sentinel-2 Level-2A includes a **Scene Classification Layer (SCL)** band.
We use SCL to mask clouds, cloud shadows, snow, and other unwanted pixels.

| SCL Value | Class             | Keep? |
|-----------|-------------------|-------|
| 4         | Vegetation        | Yes   |
| 5         | Bare soil         | Yes   |
| 6         | Water             | Yes   |
| 7         | Unclassified      | Yes   |
| 3, 8, 9, 10, 11 | Cloud/shadow/snow | No |

**NDVI = (NIR - Red) / (NIR + Red) = (B8 - B4) / (B8 + B4)**

> **HINT:** B8 = NIR, B4 = Red — same as W8. The `normalizedDifference` function
> takes `['NIR_band', 'Red_band']` in that order.

In [ ]:
# ============================================================
# S3 — NDVI Calculation + Cloud Masking — EXERCISE
# ============================================================

def mask_and_ndvi(image):
    """Apply SCL cloud mask and compute NDVI for a single S2 image."""
    scl = image.select('SCL')

    # TODO: Create a mask that keeps good pixels (vegetation=4, bare soil=5, water=6, unclassified=7)
    # HINT: Use scl.eq(value) and chain with .Or()
    # Example pattern: scl.eq(4).Or(scl.eq(5)).Or(...)
    good_mask = (scl.eq(____).Or(scl.eq(____)).Or(scl.eq(____)).Or(scl.eq(____)))

    # TODO: Compute NDVI using normalizedDifference
    # HINT: normalizedDifference takes ['NIR_band_name', 'Red_band_name']
    # For Sentinel-2: NIR = B8, Red = B4
    ndvi = image.normalizedDifference(['____', '____']).rename('NDVI')

    return (image.addBands(ndvi)
            .updateMask(good_mask)
            .copyProperties(image, ['system:time_start']))

# Apply to entire collection
ndvi_col = s2_col.map(mask_and_ndvi)

# --- Display a sample NDVI image (do not modify below) ---
sample_img = ndvi_col.sort('system:time_start').first()
vis_ndvi = {'bands': ['NDVI'], 'min': 0, 'max': 0.8,
            'palette': ['red', 'yellow', 'green', 'darkgreen']}

Map3 = geemap.Map(center=[23.70, 121.40], zoom=11)
Map3.addLayer(sample_img.select('NDVI').clip(aoi), vis_ndvi, 'Sample NDVI')
Map3.addLayer(aoi, {'color': 'red'}, 'AOI')
Map3

---
## S4 — Monthly NDVI Time Series with Spatial Spread (月均 NDVI 時間序列 + 空間分布) ✏️ EXERCISE

We aggregate NDVI to monthly medians, then retrieve **three spatial statistics**
over our AOI: the **mean** (regional average), **min** (worst pixel), and
**max** (best pixel).

### Why Three Lines Instead of One? (為什麼需要三條線？)

Imagine a classroom of 30 students taking an exam. If the class average is 75,
you might think "the class did OK." But what if the range is 20 to 100?
That tells a very different story — some students excelled, others struggled.

The same logic applies to satellite pixels:
- **Mean** = class average (overall health of the region)
- **Min** = lowest-scoring student (worst-damaged pixel — a landslide scar?)
- **Max** = top student (healthiest forest pixel — undamaged reference)

**Your tasks:**
1. Run the provided function to compute monthly mean/min/max
2. Create the matplotlib time series plot with a **shaded band** between min and max
3. Add the earthquake marker on 2024-04-03
4. Answer the guided scientific inquiry questions below

> The earthquake struck on **2024-04-03** — does the min value reveal damage
> signals that the mean alone might miss?


In [ ]:
# ============================================================
# S4 — Monthly NDVI Time Series with Spatial Spread — PARTIALLY COMPLETE
# ============================================================

# --- This function is COMPLETE — do not modify ---
def compute_monthly_ndvi(ndvi_collection, aoi, start_year=2020, end_year=2026):
    """Compute monthly NDVI statistics (mean, min, max) over an AOI.
    Returns a list of (date, mean, min, max) tuples.
    """
    results = []

    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            start = f'{year}-{month:02d}-01'
            if month == 12:
                end = f'{year + 1}-01-01'
            else:
                end = f'{year}-{month + 1:02d}-01'

            monthly = ndvi_collection.filterDate(start, end).select('NDVI')
            n = monthly.size().getInfo()

            if n == 0:
                results.append((datetime(year, month, 15), None, None, None))
                continue

            median_img = monthly.median()
            stats = median_img.reduceRegion(
                reducer=ee.Reducer.mean()
                    .combine(ee.Reducer.min(), sharedInputs=True)
                    .combine(ee.Reducer.max(), sharedInputs=True),
                geometry=aoi,
                scale=100,
                maxPixels=1e8
            ).getInfo()

            v_mean = stats.get('NDVI_mean')
            v_min  = stats.get('NDVI_min')
            v_max  = stats.get('NDVI_max')
            results.append((datetime(year, month, 15), v_mean, v_min, v_max))

            if v_mean is not None:
                print(f'  {start[:7]}: mean={v_mean:.3f}  min={v_min:.3f}  '
                      f'max={v_max:.3f}  (n={n})')
            else:
                print(f'  {start[:7]}: no data  (n={n})')

    return results

print('Computing monthly NDVI with spatial statistics...')
print('(mean = regional average, min = worst pixel, max = best pixel)')
print()
monthly_data = compute_monthly_ndvi(ndvi_col, aoi)

# --- Prepare data for plotting (do not modify) ---
dates_plot  = [d for d, m, mn, mx in monthly_data if m is not None]
mean_plot   = [m for d, m, mn, mx in monthly_data if m is not None]
min_plot    = [mn for d, m, mn, mx in monthly_data if m is not None]
max_plot    = [mx for d, m, mn, mx in monthly_data if m is not None]


In [ ]:
# ============================================================
# S4 (continued) — Plot NDVI Time Series with Spread — EXERCISE
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 9), height_ratios=[3, 1],
                         sharex=True, gridspec_kw={'hspace': 0.08})

# --- Upper panel: NDVI time series with min-max band ---
ax = axes[0]

# TODO: Add a shaded band between min_plot and max_plot
# HINT: ax.fill_between(x, y_low, y_high, alpha=0.2, color='green',
#                       label='Spatial range (min-max)')
ax.fill_between(dates_plot, ____, ____,
                alpha=0.2, color='green', label='Spatial range (min-max)')

# Plot mean, min, max lines (complete)
ax.plot(dates_plot, mean_plot, 'o-', color='green', markersize=4, linewidth=1.5,
        label='Spatial mean (AOI average)', zorder=3)
ax.plot(dates_plot, min_plot, '--', color='brown', linewidth=1, alpha=0.7,
        label='Spatial min (worst pixel)')
ax.plot(dates_plot, max_plot, '--', color='darkgreen', linewidth=1, alpha=0.7,
        label='Spatial max (best pixel)')

# TODO: Add the earthquake marker as a vertical dashed red line
# HINT: datetime(2024, 4, 3)
eq_date = datetime(2024, 4, 3)
ax.axvline(eq_date, color='red', linestyle='--', linewidth=2,
           label='Earthquake 2024-04-03')
ax.annotate('Hualien Earthquake\n(花蓮地震 Mw 7.4)',
            xy=(eq_date, max(max_plot) * 0.92),
            xytext=(eq_date + timedelta(days=60), max(max_plot) * 0.96),
            fontsize=10, color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

ax.set_ylabel('NDVI', fontsize=12)
ax.set_title('Hualien NDVI Time Series with Spatial Spread (2020-2026)\n'
             '花蓮 NDVI 時間序列（含空間分布範圍）', fontsize=14)
ax.legend(loc='lower left', fontsize=9)
ax.grid(True, alpha=0.3)

# --- Lower panel: Spread (max - min) ---
ax2 = axes[1]
spread_plot = [mx - mn for mn, mx in zip(min_plot, max_plot)]
ax2.bar(dates_plot, spread_plot, width=25, color='orange', alpha=0.7,
        label='Spatial spread (max - min)')
ax2.axvline(eq_date, color='red', linestyle='--', linewidth=2)
ax2.set_ylabel('Spread', fontsize=12)
ax2.set_xlabel('Date (日期)', fontsize=12)
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- Summary statistics ---
pre_spreads = [mx - mn for d, m, mn, mx in monthly_data
               if m is not None and d < eq_date]
post_spreads = [mx - mn for d, m, mn, mx in monthly_data
                if m is not None and d >= eq_date]
if pre_spreads and post_spreads:
    print(f'  Pre-earthquake  avg spread: {sum(pre_spreads)/len(pre_spreads):.4f}')
    print(f'  Post-earthquake avg spread: {sum(post_spreads)/len(post_spreads):.4f}')


### S4 — Scientific Inquiry: From Data to Discovery (科學探究：從數據到發現)

Good scientists don't just plot data — they **question what the data is telling them**.
Answer these questions based on your three-line plot above.

---

**Q1: Observe (觀察) — Describe the three lines:**
- Does the mean (green) show a clear earthquake signal? Why or why not?
- Does the min (brown) show a different story? What happens to it after 2024-04?
- Does the max (dark green) change much after the earthquake?

> *Your answer:*

---

**Q2: Question (質疑方法論) — Why does the mean miss the signal?**

Our AOI is approximately 24 km x 10 km. If earthquake landslides cover
5 km^2 of this area, what percentage of pixels are damaged? Would the
mean NDVI shift noticeably? What does this tell you about choosing the
**right spatial scale** for analysis?

> *Your answer:*

---

**Q3: Interpret the spread (解讀展幅) — Look at the orange bar chart:**
- Did the spread (max - min) change after the earthquake?
- What does an increase in spread mean physically? (Think: what happens to a
  landscape when some areas collapse into landslides while nearby forests are fine?)

> *Your answer:*

---

**Q4: Design the next experiment (設計下一步實驗):**

If you wanted to find **exactly where** the worst-damaged pixels are,
what would you do next? (Hint: think about what D5 does, or how you might
use a smaller AOI focused on known landslide areas.)

> *Your answer:*

---

**Q5: Connect to the real world (連結真實世界):**

Imagine you work for the Soil and Water Conservation Bureau (水土保持局).
After a major earthquake, you need to quickly identify which mountain roads
and villages are at risk from landslides. Based on what you've learned:
- Would you use mean NDVI or min NDVI to prioritize rescue efforts? Why?
- How would you combine optical (NDVI) and SAR data for a cloudy area like Hualien?
- What other data would you want? (Elevation? Slope? Population density?)

> *Your answer:*

---

> **The scientific process (科學方法):**
> Observe anomaly → Question methodology → Refine analysis → Form hypothesis → Test it
>
> This is exactly what we just did: the mean looked "fine" → we questioned whether
> averaging was appropriate → we added min/max → we saw the real damage signal → 
> now D5 will map exactly where the damage occurred.


---
## S4b — Scale Test: Taroko Gorge vs Broad Hualien (尺度測試：太魯閣 vs 寬幅花蓮) ✏️ EXERCISE

You just analyzed NDVI across a broad AOI (~240 km^2) that includes flatlands,
wetlands, and mountains. But the 2024 earthquake damage was concentrated in the
**Taroko Gorge** — steep mountain slopes where landslides buried roads and trails.

**Hypothesis:** If we focus on the damage zone, the NDVI drop will be much clearer.

**Your task:** Run the same `compute_monthly_ndvi` function on a smaller,
focused AOI and compare the results side-by-side.

| AOI | Area | What's Inside |
|-----|------|---------------|
| Broad Hualien | ~240 km^2 | Plains, wetlands, city, mountains |
| Taroko Focus | ~1,040 km² | Taroko Gorge, Suhua Hwy, mountain slopes |


In [ ]:
# ============================================================
# S4b — Taroko Gorge Focused Analysis — EXERCISE
# ============================================================

# TODO: Define the Taroko focused AOI
# This covers the steep gorge area where landslides were most severe
# HINT: The coordinates are [west, south, east, north]
TAROKO_BBOX = [121.34526379253053, 24.046021742135874, 121.85149217685861, 24.35767637905926]
aoi_taroko = ee.Geometry.Rectangle(TAROKO_BBOX)

# --- Compute monthly NDVI for Taroko (reuse the same function) ---
print('Computing monthly NDVI for Taroko Gorge focused AOI...')
print(f'Taroko BBOX: {TAROKO_BBOX}')
print()
taroko_data = compute_monthly_ndvi(ndvi_col, aoi_taroko)

# --- Prepare Taroko data ---
t_dates = [d for d, m, mn, mx in taroko_data if m is not None]
t_mean  = [m for d, m, mn, mx in taroko_data if m is not None]
t_min   = [mn for d, m, mn, mx in taroko_data if m is not None]
t_max   = [mx for d, m, mn, mx in taroko_data if m is not None]

# ============================================================
# Side-by-side comparison
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10),
                         height_ratios=[3, 1],
                         sharex=True,
                         gridspec_kw={'hspace': 0.08, 'wspace': 0.25})

eq_date = datetime(2024, 4, 3)

for col, (title, dates, means, mins, maxs, desc) in enumerate([
    ('Broad Hualien AOI (~240 km\u00b2)',
     dates_plot, mean_plot, min_plot, max_plot,
     'Mixed terrain: plains + wetlands + mountains'),
    ('Taroko Gorge Focus (~1,040 km²)',
     t_dates, t_mean, t_min, t_max,
     'Taroko Gorge to Suhua Hwy mountain corridor'),
]):
    ax = axes[0, col]

    # TODO: Add the shaded band between min and max
    # HINT: Same as S4 — ax.fill_between(dates, mins, maxs, ...)
    ax.fill_between(dates, ____, ____,
                    alpha=0.2, color='green', label='Range (min-max)')

    ax.plot(dates, means, 'o-', color='green', markersize=3,
            linewidth=1.5, label='Mean', zorder=3)
    ax.plot(dates, mins, '--', color='brown', linewidth=1,
            alpha=0.7, label='Min (worst pixel)')
    ax.plot(dates, maxs, '--', color='darkgreen', linewidth=1,
            alpha=0.7, label='Max (best pixel)')
    ax.axvline(eq_date, color='red', linestyle='--', linewidth=2)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('NDVI' if col == 0 else '')
    ax.legend(fontsize=8, loc='lower left')
    ax.grid(True, alpha=0.3)
    ax.text(0.5, -0.02, desc, transform=ax.transAxes,
            ha='center', fontsize=10, color='gray', style='italic')
    ax.set_ylim(-0.5, 1.05)

    ax2 = axes[1, col]
    spreads = [mx - mn for mn, mx in zip(mins, maxs)]
    ax2.bar(dates, spreads, width=25, color='orange', alpha=0.7)
    ax2.axvline(eq_date, color='red', linestyle='--', linewidth=2)
    ax2.set_ylabel('Spread' if col == 0 else '')
    ax2.grid(True, alpha=0.3)

for ax in axes[1, :]:
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.set_xlabel('Date (日期)')

fig.suptitle('Scale Matters: Broad AOI vs Taroko Focus\n'
             '尺度效應：寬幅 AOI vs 太魯閣聚焦', fontsize=15, y=1.01)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- Summary stats ---
for label, data in [('Broad Hualien', monthly_data), ('Taroko Focus', taroko_data)]:
    pre  = [(d, m, mn, mx) for d, m, mn, mx in data
            if m is not None and d < eq_date]
    post = [(d, m, mn, mx) for d, m, mn, mx in data
            if m is not None and d >= eq_date]
    if pre and post:
        pre_mean = sum(m for _, m, _, _ in pre) / len(pre)
        post_mean = sum(m for _, m, _, _ in post) / len(post)
        pre_min_avg = sum(mn for _, _, mn, _ in pre) / len(pre)
        post_min_avg = sum(mn for _, _, mn, _ in post) / len(post)
        print(f'\n  {label}:')
        print(f'    Mean  — pre: {pre_mean:.4f}  post: {post_mean:.4f}  '
              f'change: {post_mean-pre_mean:+.4f}')
        print(f'    Min   — pre: {pre_min_avg:.4f}  post: {post_min_avg:.4f}  '
              f'change: {post_min_avg-pre_min_avg:+.4f}')


### S4b — Your Analysis (你的分析)

**Q1: Compare the two panels.** Which AOI shows a clearer earthquake damage signal?
Look at the mean line, the min line, and the spread — which changed more dramatically
in the Taroko focus?

> *Your answer:*

**Q2: Why does scale matter?** If a government agency analyzed the entire Hualien
county and concluded "vegetation impact was minimal," would that be accurate?
What would you recommend they do differently?

> *Your answer:*

**Q3: Hierarchical analysis.** Real disaster monitoring uses three levels:
1. Broad scan (country/county) → detect anomalies
2. Focused zoom (valley/slope) → pinpoint damage
3. Pixel-level mapping → individual landslide scars

You just performed levels 1 and 2. Which section of this notebook performs level 3?

> *Your answer:*

> **Key takeaway:** The same satellite data, analyzed at different spatial scales,
> tells fundamentally different stories. Choosing the right scale is as important
> as choosing the right algorithm.


---
## S5 — Three-Phase Composite Comparison (三期中值合成比較) ✏️ EXERCISE

The 2024 earthquake triggered landslides that created **landslide dams** (堰塞湖),
including the Mataian landslide dam (馬太鞍堰塞湖). To track this multi-event
disaster timeline, we compare three periods:

| Phase | Period | What Happened |
|-------|--------|---------------|
| **Phase 1** | 2023/01 – 2024/03 | Pre-earthquake baseline |
| **Phase 2** | 2024/04 – 2024/09 | Post-earthquake: landslides, road collapse |
| **Phase 3** | 2025/10 – 2026/03 | Post-landslide-dam: breach aftermath |

> **HINT:** Use `.filterDate('start', 'end').select('NDVI').median()` for each phase.


In [ ]:
# ============================================================
# S5 — Three-Phase Composite Comparison — EXERCISE
# ============================================================

# TODO: Create Phase 1 — pre-earthquake NDVI composite
# HINT: filterDate('2023-01-01', '2024-03-31'), select('NDVI'), .median()
pre_ndvi = (ndvi_col
            .filterDate('____', '____')
            .select('NDVI')
            .median())

# TODO: Create Phase 2 — post-earthquake NDVI composite
# HINT: filterDate('2024-04-01', '2024-09-30')
post_eq_ndvi = (ndvi_col
                .filterDate('____', '____')
                .select('NDVI')
                .median())

# TODO: Create Phase 3 — post-landslide-dam NDVI composite
# HINT: filterDate('2025-10-01', '2026-03-31')
post_dam_ndvi = (ndvi_col
                 .filterDate('____', '____')
                 .select('NDVI')
                 .median())

# TODO: Compute change maps
# HINT: .subtract() and .rename()
delta_eq    = post_eq_ndvi.subtract(____).rename('delta_NDVI')
delta_dam   = post_dam_ndvi.subtract(____).rename('delta_NDVI_dam')
delta_total = post_dam_ndvi.subtract(____).rename('delta_NDVI_total')

# --- Visualization (provided — do not modify below) ---
vis_ndvi  = {'min': 0, 'max': 0.8,
             'palette': ['red', 'yellow', 'green', 'darkgreen']}
vis_delta = {'min': -0.3, 'max': 0.3,
             'palette': ['red', 'white', 'blue']}

Map5 = geemap.Map(center=[23.70, 121.40], zoom=11)
Map5.addLayer(pre_ndvi.clip(aoi), vis_ndvi, 'Phase 1: Pre-EQ')
Map5.addLayer(post_eq_ndvi.clip(aoi), vis_ndvi, 'Phase 2: Post-EQ')
Map5.addLayer(post_dam_ndvi.clip(aoi), vis_ndvi, 'Phase 3: Post-Dam')
Map5.addLayer(delta_eq.clip(aoi), vis_delta, 'ΔNDVI: EQ damage (P2-P1)')
Map5.addLayer(delta_dam.clip(aoi), vis_delta, 'ΔNDVI: Dam change (P3-P2)')
Map5.addLayer(delta_total.clip(aoi), vis_delta, 'ΔNDVI: Total (P3-P1)')
Map5.addLayer(aoi, {'color': 'yellow'}, 'AOI')
Map5


### S5 Visual — 三期衛星影像對比 ✏️ EXERCISE

用 geemap 在互動地圖上展示三期的 RGB 真色合成和 NDVI 合成。

**觀察任務：**
1. 在 RGB 地圖上，找到太魯閣峽谷附近最明顯的崩塌區（Phase 1 綠 → Phase 2 灰白）
2. 在 ΔNDVI 地圖上，比較三張差值圖的紅色（損失）和綠色（恢復）分佈
3. 堰塞湖潰堤後是否造成新損害？還是已有部分恢復？

In [ ]:
# ============================================================
# S5 Visual — Three-Phase Map Comparison — EXERCISE
# ============================================================

# TODO: Build cloud-masked S2 RGB composites for three phases
s2_masked = s2_col.map(lambda img: img.updateMask(
    img.select('SCL').eq(4).Or(img.select('SCL').eq(5))
    .Or(img.select('SCL').eq(6)).Or(img.select('SCL').eq(7))
))

rgb_pre = s2_masked.filterDate('2023-01-01', '2024-03-31').median().select(['B4','B3','B2'])
rgb_post_eq = s2_masked.filterDate('____', '____').median().select(['B4','B3','B2'])  # TODO: fill dates
rgb_post_dam = s2_masked.filterDate('____', '____').median().select(['B4','B3','B2'])  # TODO: fill dates

rgb_vis = {'min': 0, 'max': 3000}
ndvi_vis = {'min': 0, 'max': 0.8, 'palette': ['brown', 'yellow', 'green', 'darkgreen']}
delta_vis = {'min': -0.3, 'max': 0.3, 'palette': ['red', 'orange', 'white', 'lightgreen', 'darkgreen']}

# === Map 1: RGB True Color ===
Map1 = geemap.Map(center=[24.20, 121.60], zoom=11)
Map1.addLayer(rgb_pre, rgb_vis, 'Phase 1: Pre-EQ RGB')
Map1.addLayer(rgb_post_eq, rgb_vis, 'Phase 2: Post-EQ RGB')
Map1.addLayer(rgb_post_dam, rgb_vis, 'Phase 3: Post-Dam RGB')
Map1.addLayerControl()
Map1


In [ ]:
# === Map 2: NDVI + Delta Maps ===
# TODO: Create a geemap Map and add the six layers:
#   - pre_ndvi, post_eq_ndvi, post_dam_ndvi (NDVI composites)
#   - delta_eq, delta_dam, delta_total (difference maps)

Map2 = geemap.Map(center=[24.20, 121.60], zoom=11)
# TODO: Add your layers here using Map2.addLayer(...)
#   HINT: use ndvi_vis for NDVI layers, delta_vis for difference layers
#   HINT: use shown=False for layers you don't want visible by default

Map2.addLayerControl()
Map2


**✏️ 觀察紀錄：**

1. 地震直接損害（delta_eq 紅色區）主要集中在哪些地形？（峽谷？稜線？河谷？）

   你的觀察：___

2. 堰塞湖潰堤後（delta_dam），是出現新損害還是植被恢復？分佈在哪裡？

   你的觀察：___

3. 如果你是水土保持局工程師，這三張 ΔNDVI 地圖分別幫你做什麼決策？

   你的想法：___

In [ ]:
# --- Three-phase statistics (do not modify) ---
for label, delta_img, band_name in [
    ('EQ damage (Phase 2 - Phase 1)',    delta_eq,    'delta_NDVI'),
    ('Dam change (Phase 3 - Phase 2)',   delta_dam,   'delta_NDVI_dam'),
    ('Total change (Phase 3 - Phase 1)', delta_total, 'delta_NDVI_total'),
]:
    stats = delta_img.reduceRegion(
        reducer=ee.Reducer.mean()
            .combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=aoi, scale=100, maxPixels=1e8
    ).getInfo()
    mean_v = stats.get(f'{band_name}_mean')
    std_v  = stats.get(f'{band_name}_stdDev')
    damage = delta_img.lt(-0.15)
    area = damage.multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi,
        scale=100, maxPixels=1e8
    ).getInfo()
    area_km2 = (area.get(band_name, 0) or 0) / 1e6
    print(f'\n  {label}:')
    if mean_v: print(f'    Mean ΔNDVI:  {mean_v:+.4f}')
    if std_v:  print(f'    StdDev:      {std_v:.4f}')
    print(f'    Damaged area (ΔNDVI < -0.15): {area_km2:.2f} km2')

# Keep reference for S7
delta_ndvi = delta_eq


---
## S6 — Sentinel-1 SAR GRD Time Series (Sentinel-1 SAR 時間序列) ✏️ EXERCISE

Synthetic Aperture Radar (SAR) is **cloud-independent** — it penetrates clouds and
works day/night. This is critical for Hualien, which has frequent cloud cover.

We use **Sentinel-1 GRD** (Ground Range Detected) data:
- **IW mode** (Interferometric Wide swath) — standard land observation mode
- **VV polarization** — sensitive to surface roughness changes
- Values are in **decibels (dB)** — logarithmic scale

> **HINT:** The instrument mode is called `'IW'` and we want the `'VV'` band.

In [ ]:
# ============================================================
# S6 — Sentinel-1 SAR GRD Time Series — EXERCISE
# ============================================================

s1_col = (ee.ImageCollection('COPERNICUS/S1_GRD')
          .filterDate('2022-01-01', '2026-03-31')
          # TODO: Filter by instrument mode — should be 'IW'
          # HINT: ee.Filter.eq('instrumentMode', 'mode_name')
          .filter(ee.Filter.eq('instrumentMode', '____'))
          .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .filterBounds(aoi)
          # TODO: Select the VV band
          # HINT: .select('band_name')
          .select('____'))

s1_count = s1_col.size().getInfo()
print(f"  Sentinel-1 GRD images (2022-2026, DESC, VV): {s1_count}")

In [ ]:
# ============================================================
# S6 (continued) — SAR Time Series Extraction & Plot — COMPLETE
# ============================================================

# --- Compute mean VV per image over AOI (complete) ---
def get_vv_stats(image):
    """Extract mean VV over AOI for a single image."""
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=100,
        maxPixels=1e8
    )
    return image.set('mean_VV', stats.get('VV'))

s1_with_stats = s1_col.map(get_vv_stats)

# Retrieve client-side
s1_info = s1_with_stats.aggregate_array('system:time_start').getInfo()
s1_vv = s1_with_stats.aggregate_array('mean_VV').getInfo()

s1_dates = [datetime.utcfromtimestamp(t / 1000) for t in s1_info]
s1_values = [v for v in s1_vv]

# Filter out None values
valid = [(d, v) for d, v in zip(s1_dates, s1_values) if v is not None]
s1_dates_clean = [d for d, v in valid]
s1_values_clean = [v for d, v in valid]

print(f"  Valid observations: {len(s1_dates_clean)}")

# --- Plot ---
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(s1_dates_clean, s1_values_clean, '.', color='navy', markersize=3, alpha=0.6,
        label='Mean VV (dB)')

# Rolling average
df_s1 = pd.DataFrame({'date': s1_dates_clean, 'VV': s1_values_clean})
df_s1 = df_s1.sort_values('date')
df_s1['VV_smooth'] = df_s1['VV'].rolling(window=5, center=True).mean()
ax.plot(df_s1['date'], df_s1['VV_smooth'], '-', color='dodgerblue', linewidth=2,
        label='5-image rolling mean')

# TODO: Add the earthquake marker — same as S4
# HINT: ax.axvline(datetime(2024, 4, 3), color='red', linestyle='--', linewidth=2)
# ---- YOUR CODE HERE ----


ax.set_xlabel('Date (日期)', fontsize=12)
ax.set_ylabel('Mean VV Backscatter (dB)', fontsize=12)
ax.set_title('Hualien SAR VV Time Series (2022-2026)\n花蓮 SAR VV 時間序列', fontsize=14)
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### S6 — Your Observations (你的觀察)

**TODO:** After running the SAR time series plot, write 2-3 observations:

1. *(How does SAR data availability compare to optical (Sentinel-2)?)*
2. *(Do you see any change around the earthquake date?)*
3. *(Why is SAR particularly useful for monitoring Hualien?)*

---
## S7 — SAR Pre/Post Composite + Cross-Reference (SAR 震前震後 + 交叉比對) ✏️ EXERCISE

Similar to S5, create pre/post median composites from SAR data and compute
the difference (Delta-VV). Then cross-reference with Delta-NDVI to identify areas
where **both** optical and SAR sensors detect damage.

**High-confidence damage = areas where BOTH sensors agree:**
- Delta-NDVI < -0.15 (vegetation loss detected by optical)
- |Delta-VV| > 2 dB (backscatter change detected by SAR)

> **HINT:** Follow the same pattern as S5: filterDate -> median -> subtract.

In [ ]:
# ============================================================
# S7 — SAR Pre/Post Composite + Cross-Reference — EXERCISE
# ============================================================

# TODO: Create pre-earthquake VV composite (same date range as S5)
# HINT: s1_col.filterDate('2023-01-01', '2024-03-31').median()
pre_vv = (s1_col
          .filterDate('____', '____')
          .median())

# TODO: Create post-earthquake VV composite
# HINT: s1_col.filterDate('2024-04-01', '2026-03-31').median()
post_vv = (s1_col
           .filterDate('____', '____')
           .median())

# TODO: Compute delta_VV = post - pre
delta_vv = ____.subtract(____).rename('delta_VV')

# --- Visualization (provided) ---
vis_vv = {'min': -20, 'max': -5, 'palette': ['black', 'white']}
vis_dvv = {'min': -4, 'max': 4, 'palette': ['blue', 'white', 'red']}

Map7 = geemap.Map(center=[23.70, 121.40], zoom=11)
Map7.addLayer(pre_vv.clip(aoi), vis_vv, 'Pre-EQ VV (震前)')
Map7.addLayer(post_vv.clip(aoi), vis_vv, 'Post-EQ VV (震後)')
Map7.addLayer(delta_vv.clip(aoi), vis_dvv, 'Delta-VV (Post - Pre)')
Map7.addLayer(aoi, {'color': 'yellow'}, 'AOI')
Map7

In [ ]:
# ============================================================
# S7 (continued) — Cross-Reference Analysis — EXERCISE
# ============================================================

# TODO: Create a mask where BOTH conditions are true:
# 1. delta_NDVI < -0.15 (vegetation loss)
# 2. abs(delta_VV) > 2 (SAR backscatter change > 2 dB)
#
# HINT: For condition 1, use delta_ndvi.lt(-0.15)
# HINT: For condition 2, use delta_vv.abs().gt(2)
# HINT: Combine with .And() and then .selfMask()

ndvi_damage = ____
sar_damage = ____
high_confidence = ndvi_damage.And(sar_damage).selfMask()

# --- Visualization (provided) ---
vis_delta = {'min': -0.3, 'max': 0.3, 'palette': ['red', 'white', 'blue']}

Map7b = geemap.Map(center=[23.70, 121.40], zoom=11)
Map7b.addLayer(delta_ndvi.clip(aoi), vis_delta, 'Delta-NDVI')
Map7b.addLayer(delta_vv.clip(aoi), vis_dvv, 'Delta-VV')
Map7b.addLayer(high_confidence.clip(aoi),
               {'palette': ['magenta'], 'min': 0, 'max': 1},
               'High-Confidence Damage (Delta-NDVI<-0.15 & |Delta-VV|>2dB)')
Map7b.addLayer(aoi, {'color': 'yellow'}, 'AOI')
Map7b

In [ ]:
# --- Damage area statistics (do not modify) ---
combined_area = high_confidence.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=100,
    maxPixels=1e8
).getInfo()

# The band name depends on which image was used in .And()
# Try both possible band names
combined_km2 = 0
for key in combined_area:
    if combined_area[key] is not None:
        combined_km2 = combined_area[key] / 1e6
        break

print("Cross-sensor Damage Detection (跨感測器損害偵測):")
print(f"  Area detected by BOTH sensors: {combined_km2:.2f} km2")
print()
print("  Multi-sensor fusion provides higher confidence in damage assessment.")
print("  Optical detects vegetation loss; SAR detects surface roughness change.")
print("  Their intersection = high-confidence damage zones.")

---
## S8 — Export GeoTIFF (匯出 GeoTIFF) ✏️ EXERCISE

GEE analysis results live in the cloud. To use them in local software (QGIS, Python
classification from W12), we export to Google Drive as GeoTIFF files.

This is the bridge between cloud-scale processing and local analysis.

> **HINT:** Sentinel-2 native resolution is 10m. The CRS for UTM Zone 51N
> (covering eastern Taiwan) is `'EPSG:32651'`.

In [ ]:
# ============================================================
# S8 — Export GeoTIFF — EXERCISE
# ============================================================

# --- Export delta_NDVI ---
task_delta = ee.batch.Export.image.toDrive(
    image=delta_ndvi.clip(aoi).toFloat(),
    description='Hualien_DeltaNDVI',
    folder='GEE_Exports',
    fileNamePrefix='hualien_delta_ndvi',
    region=aoi,
    # TODO: Set the spatial resolution in meters
    # HINT: Sentinel-2 native resolution is 10 meters
    scale=____,
    # TODO: Set the coordinate reference system
    # HINT: UTM Zone 51N for eastern Taiwan = 'EPSG:32651'
    crs='____',
    maxPixels=1e9
)
task_delta.start()
print("  Task started: Hualien_DeltaNDVI")

# --- Export post-earthquake NDVI composite ---
task_ndvi = ee.batch.Export.image.toDrive(
    image=post_eq_ndvi.clip(aoi).toFloat(),
    description='Hualien_PostEQ_NDVI',
    folder='GEE_Exports',
    fileNamePrefix='hualien_post_eq_ndvi',
    region=aoi,
    # TODO: Same scale and CRS as above
    scale=____,
    crs='____',
    maxPixels=1e9
)
task_ndvi.start()
print("  Task started: Hualien_PostEQ_NDVI")

print()
print("=" * 60)
print("  Export tasks submitted to Google Earth Engine")
print("  Check status at: https://code.earthengine.google.com/tasks")
print("  Files will appear in Google Drive -> GEE_Exports folder")
print("=" * 60)

---
## S10 — Creative Exploration: Your Own Time Series (自由探索：你的時間序列) 🎨 EXERCISE

Now it's YOUR turn! Choose a **location** and a **spectral index** that interests you,
and build a monthly time series using the same GEE workflow you just learned.

### Rules (規則)
1. **Choose a different study area** — your hometown, a place you've traveled, or a location in the news
2. **Choose a spectral index** from the table below (can be NDVI again, but must be a different area)
3. **Time range:** at least 2 years
4. **Mark at least one "event"** on the plot (typhoon, earthquake, construction, fire, season change, etc.)
5. **Write a 100-word explanation** of why you chose this location + index, and what the time series reveals

### Available Indices (可選指標)

| Index | Formula (Sentinel-2 bands) | Detects | Best For |
|-------|---------|---------|----------|
| **NDVI** | (B8-B4)/(B8+B4) | Vegetation health | Forests, agriculture, post-disaster |
| **NDWI** | (B3-B8)/(B3+B8) | Water content | Floods, reservoirs, wetlands |
| **NDBI** | (B11-B8)/(B11+B8) | Built-up areas | Urbanization, construction |
| **EVI** | 2.5*(B8-B4)/(B8+6*B4-7.5*B2+1) | Enhanced vegetation | Dense forests (less saturated than NDVI) |
| **VV (SAR)** | Sentinel-1 VV band (dB) | Surface roughness | Cloud-prone areas, all-weather |

### Presentation (課堂發表)
Each student will **present their time series to the class** (~2 min per person):
- Show your plot on screen
- Explain your choice: why this location + index + event?
- Share one interesting finding

> **TIP:** The code structure is almost identical to S2-S4 — you only need to change the AOI coordinates, the band names in `normalizedDifference()`, and the date range. Copy and adapt!

In [ ]:
# ============================================================
# S10 — Step 1: Define YOUR Study Area & Index
# ============================================================

# TODO: Define your own AOI — replace with your chosen coordinates
# HINT: Use Google Maps to find the bounding box [west, south, east, north]
# Example: Taipei city center ~ [121.49, 25.01, 121.58, 25.08]
MY_BBOX = [____, ____, ____, ____]
my_aoi = ee.Geometry.Rectangle(MY_BBOX)

# TODO: Choose your index — uncomment ONE of the following, or write your own
# Option A: NDVI (vegetation)
# my_index_bands = ['B8', 'B4']
# my_index_name = 'NDVI'

# Option B: NDWI (water)
# my_index_bands = ['B3', 'B8']
# my_index_name = 'NDWI'

# Option C: NDBI (built-up)
# my_index_bands = ['B11', 'B8']
# my_index_name = 'NDBI'

my_index_bands = ['____', '____']
my_index_name = '____'

# TODO: Set your date range (at least 2 years)
MY_START = '____'  # e.g. '2020-01-01'
MY_END   = '____'  # e.g. '2024-12-31'

# TODO: Set your event date and description
MY_EVENT_DATE = '____'  # e.g. '2024-04-03'
MY_EVENT_LABEL = '____'  # e.g. 'Typhoon Gaemi'

print(f"  Study area: {MY_BBOX}")
print(f"  Index: {my_index_name} = ({my_index_bands[0]} - {my_index_bands[1]}) / ({my_index_bands[0]} + {my_index_bands[1]})")
print(f"  Period: {MY_START} to {MY_END}")
print(f"  Event: {MY_EVENT_LABEL} ({MY_EVENT_DATE})")

In [ ]:
# ============================================================
# S10 — Step 2: Build & Plot Your Time Series
# ============================================================

# --- Filter and compute index (adapts S2-S3 pattern) ---
my_s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
         .filterDate(MY_START, MY_END)
         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
         .filterBounds(my_aoi))

def my_mask_and_index(image):
    scl = image.select('SCL')
    good = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(7))
    idx = image.normalizedDifference(my_index_bands).rename(my_index_name)
    return (image.addBands(idx)
            .updateMask(good)
            .copyProperties(image, ['system:time_start']))

my_idx_col = my_s2.map(my_mask_and_index)

# --- Monthly aggregation (reuses S4 pattern) ---
print(f"Computing monthly {my_index_name} for your study area...")
start_y = int(MY_START[:4])
end_y = int(MY_END[:4])
my_results = []

for year in range(start_y, end_y + 1):
    for month in range(1, 13):
        start = f'{year}-{month:02d}-01'
        end = f'{year}-{month + 1:02d}-01' if month < 12 else f'{year + 1}-01-01'
        if start > MY_END:
            break
        monthly = my_idx_col.filterDate(start, end).select(my_index_name)
        n = monthly.size().getInfo()
        if n == 0:
            my_results.append((datetime(year, month, 15), None))
            continue
        val = monthly.median().reduceRegion(
            reducer=ee.Reducer.mean(), geometry=my_aoi,
            scale=100, maxPixels=1e8
        ).getInfo().get(my_index_name)
        my_results.append((datetime(year, month, 15), val))
        status = f'{val:.4f}' if val else 'no data'
        print(f'  {start[:7]}: {my_index_name} = {status}  (n={n})')

# --- Plot ---
my_dates = [d for d, v in my_results if v is not None]
my_vals  = [v for d, v in my_results if v is not None]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(my_dates, my_vals, 'o-', color='teal', markersize=4, linewidth=1.5,
        label=f'Monthly Median {my_index_name}')
ax.axvline(x=pd.Timestamp(MY_EVENT_DATE), color='red', linestyle='--', linewidth=2,
           label=MY_EVENT_LABEL)
ax.annotate(MY_EVENT_LABEL,
            xy=(pd.Timestamp(MY_EVENT_DATE), max(my_vals) * 0.95),
            xytext=(pd.Timestamp(MY_EVENT_DATE) + timedelta(days=45), max(my_vals) * 0.98),
            fontsize=10, color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

ax.set_xlabel('Date (日期)', fontsize=12)
ax.set_ylabel(f'Mean {my_index_name}', fontsize=12)
ax.set_title(f'Your {my_index_name} Time Series\n你的 {my_index_name} 時間序列', fontsize=14)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\n  Total months with data: {len(my_vals)}")

### S10 — Your Explanation (你的說明)

**TODO:** Write ~100 words explaining your exploration. This will be the basis of your class presentation.

**1. Why this location? (為什麼選這個地點？)**

> *Your answer:*

**2. Why this index? (為什麼選這個指標？)**

> *Your answer:*

**3. What event did you mark? What happened? (你標記了什麼事件？)**

> *Your answer:*

**4. What does the time series reveal? (時間序列揭示了什麼？)**

> *Your answer:*

**5. One interesting or surprising finding (一個有趣或意外的發現):**

> *Your answer:*

---
## S9 — Reflection Questions (反思問題) ✏️ EXERCISE

Answer the following questions based on your analysis. Write your answers directly
in this cell (double-click to edit).

---

**Q1: How many Sentinel-2 images did GEE process in this notebook? If each image
is approximately 800 MB, how long would it take to download them all on a 50 Mbps
connection?**

> *Your answer:*


---

**Q2: How does the median composite approach (S5) improve upon W9's two-scene
change detection? What are the advantages of using many images vs. just two?**

> *Your answer:*


---

**Q3: What can GEE NOT do that local analysis (W8-W12) can? Think about
specialized algorithms, custom models, and data types.**

> *Your answer:*


---

**Q4: If you exported the delta_NDVI GeoTIFF from S8 and fed it into W12's
Random Forest classifier as a feature, what would you expect? Would it improve
classification accuracy for earthquake damage? Why or why not?**

> *Your answer:*

---

---
## Notebook Complete

Once you have filled in all `TODO` blanks and answered the reflection questions,
this notebook demonstrates the full GEE cloud-analysis workflow:

```
S1: Setup -> S2: Filter S2 -> S3: Cloud Mask + NDVI -> S4: Time Series
    -> S5: Pre/Post NDVI -> S6: SAR Time Series -> S7: Cross-Reference
    -> S8: Export -> S10: Creative Exploration -> S9: Reflection
```

**Key takeaway (核心收穫):**
GEE processes hundreds of satellite images on the server — no download required.
By combining optical (NDVI) and SAR (VV) sensors, we achieve higher-confidence
damage detection than either sensor alone.

> *"From pixels to petabytes — 從像素到拍位元組"*